<!-- colab-badge -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gecko-Academy/dev3pack-cohort-2026-09/blob/main/units/en/unit0/w06-portable-packages/notebook.ipynb)


In [16]:
# Preflight: environment checks with a fix for anything missing. It never raises.
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "pyproject.toml").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / "src"))

try:
    from bootcamp_agent.preflight import preflight
except ImportError:
    if "google.colab" in sys.modules:
        # Colab starts in /content with no course in it, so fetch one. A shallow
        # clone of the COHORT repository, which is the public one; the source
        # repository is private and would ask this learner for credentials.
        import subprocess

        target = Path("/content/dev3pack")
        if not (target / "pyproject.toml").exists():
            print("Colab detected — fetching the course (about 20 seconds)…")
            subprocess.run(
                ["git", "clone", "-q", "--depth", "1",
                 "https://github.com/Gecko-Academy/dev3pack-cohort-2026-09.git", str(target)],
                check=True,
            )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "-e", str(target)], check=True
        )
        REPO_ROOT = target
        sys.path.insert(0, str(REPO_ROOT / "src"))
        import os

        os.chdir(REPO_ROOT)
        from bootcamp_agent.preflight import preflight

        print(f"ready — the course is at {REPO_ROOT}")
    else:
        print("❌ bootcamp_agent not importable -> in the repo root run: uv sync --group dev")
        print("   then pick the .venv kernel (or start Jupyter with: uv run jupyter lab)")
else:
    preflight(REPO_ROOT)

✅ Python 3.11 (need >= 3.11)
✅ kernel is the repo .venv
✅ corpus loads (6 documents)
✅ lane = fake (deterministic, offline)
ready. LIVE is the fake lane.


In [17]:
from bootcamp_agent.checks import check, review
from bootcamp_agent.hints import hint  # noqa: F401 - hint("w06-e1") when you want a nudge
import bootcamp_agent.week0_checks  # noqa: F401 — importing is what registers them

# Unit 6 — A portable package

**Week 0 · Course A, chapter 2 of 4 · about 60 minutes**

**Goal:** Say which requirement pins a version and which one floats, build a package a script can import, and give it the docstring `help()` shows.

**Why it matters:** `bootcamp_agent` is exactly this: a `src/` package with a `pyproject.toml` beside it. Building a tiny one by hand is the fastest way to read the real one.

Some cells below ship **broken on purpose**, marked `<------ EDIT THIS LINE`. Run them first and
read what happens. Debugging something wrong teaches more than filling in a blank.

## 1. Pinned, or floating

**Context.** A `requirements.txt` (or the `dependencies` table in `pyproject.toml`) lists what a
package needs. `numpy==1.15.4` asks for exactly one version. `pycodestyle>=2.4.0` sets a floor and
accepts anything above it. A bare `matplotlib` accepts anything at all. Only `==` is a pin.

**Instructions.**

1. Run the cell. `requires` is the list from the lesson; `exact` and `floating` are your answer.
2. Put under `exact` every package whose requirement pins one version, and under `floating` the rest.
3. Use package names only, no version specifiers.

**Expected output**

```
matplotlib         -> floating
numpy==1.15.4      -> exact
pycodestyle>=2.4.0 -> floating
✅ w06-e1 passed
```

In [4]:
requires = ["matplotlib", "numpy==1.15.4", "pycodestyle>=2.4.0"]

table = {
    "requires": requires,
    "exact": ["numpy"],   # <------ EDIT THIS LINE: >= is a floor, not a pin
    "floating": ["matplotlib", "pycodestyle"],          # <------ EDIT THIS LINE
}

for requirement in requires:
    kind = "exact" if "==" in requirement else "floating"
    print(f"{requirement:<19}-> {kind}")

matplotlib         -> floating
numpy==1.15.4      -> exact
pycodestyle>=2.4.0 -> floating


In [5]:
check("w06-e1", table)

✅ w06-e1 passed


True

## 2. Expose the function at the top of the package

**Context.** `my_package/utils.py` defines `we_need_to_talk`. A script can already reach it as
`my_package.utils.we_need_to_talk`. The lesson's shortcut, `my_package.we_need_to_talk`, only works
if `__init__.py` imports it. An empty `__init__.py` makes a package with nothing in it.

**Instructions.**

1. Run the cell. It builds the package in a temp directory and runs `my_script.py` from there.
2. Read the error: the package exists, the attribute does not.
3. Put the one-line import from the lesson in `INIT_LINE` and run again.

**Expected output**

```
my_package/__init__.py
my_package/utils.py
my_script.py

I <3 You!
✅ w06-e2 passed
```

In [9]:
INIT_LINE = "from .utils import we_need_to_talk"   # <------ EDIT THIS LINE: the package imports nothing yet

import subprocess
import sys
import tempfile
from pathlib import Path

work_dir = Path(tempfile.mkdtemp())
package_dir = work_dir / "my_package"
package_dir.mkdir()

# work_dir/my_package/utils.py
(package_dir / "utils.py").write_text('''\
def we_need_to_talk(break_up=False):
    """Helper for communicating with partner"""
    if break_up:
        print("It's not you, it's me...")
    else:
        print('I <3 You!')
''')

# work_dir/my_package/__init__.py
(package_dir / "__init__.py").write_text(INIT_LINE)

# work_dir/my_script.py
(work_dir / "my_script.py").write_text('''\
# Import custom package
import my_package

# Realise you found your soulmate
my_package.we_need_to_talk(break_up=False)
''')


def run_script(script):
    """Run a script from work_dir, the way a user of the package would."""
    result = subprocess.run([sys.executable, script.name], cwd=work_dir,
                            capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())


for path in sorted(work_dir.rglob("*.py")):
    print(path.relative_to(work_dir))
print()
run_script(work_dir / "my_script.py")

my_package/__init__.py
my_package/utils.py
my_script.py

I <3 You!


In [10]:
check("w06-e2", work_dir)

✅ w06-e2 passed


True

## 3. The docstring help() shows

**Context.** `help(my_package)` prints NAME, PACKAGE CONTENTS and FILE. The first line of the
package docstring appears beside NAME, and the rest under DESCRIPTION, but only when `__init__.py`
opens with a docstring. A package with no description makes every user read its source to learn
what it is for.

**Instructions.**

1. Run the cell. `help()` prints the package with no DESCRIPTION.
2. Write a full sentence in `PACKAGE_DOC`: what the package is for, in words a user needs. A
   blank line (`

`) and a second sentence become the DESCRIPTION.
3. Run again and find your sentences in the output.

**Expected output**

```
Help on package my_package:

NAME
    my_package - Tools for talking to a partner.

DESCRIPTION
    One function, two moods.

PACKAGE CONTENTS
    utils

FILE
    /tmp/.../my_package/__init__.py
✅ w06-e3 passed
```

In [13]:
PACKAGE_DOC = "One function, two moods."   # <------ EDIT THIS LINE: help() has nothing to show

import subprocess
import sys
import tempfile
from pathlib import Path

documented_dir = Path(tempfile.mkdtemp())
(documented_dir / "my_package").mkdir()

(documented_dir / "my_package" / "utils.py").write_text('''\
def we_need_to_talk(break_up=False):
    """Helper for communicating with partner"""
    if break_up:
        print("It's not you, it's me...")
    else:
        print('I <3 You!')
''')

# The docstring is the first statement in __init__.py, then the import.
(documented_dir / "my_package" / "__init__.py").write_text(
    '"""' + PACKAGE_DOC + '"""\n\nfrom .utils import we_need_to_talk\n'
)

result = subprocess.run([sys.executable, "-c", "import my_package; help(my_package)"],
                        cwd=documented_dir, capture_output=True, text=True)
print((result.stdout + result.stderr).strip())

Help on package my_package:

NAME
    my_package - One function, two moods.

PACKAGE CONTENTS
    utils

FILE
    /private/var/folders/by/2lc47gmn38j18vyctj564dh40000gn/T/tmpqpyv4buh/my_package/__init__.py


In [14]:
check("w06-e3", documented_dir)

✅ w06-e3 passed


True

## Review

The scorecard for this unit. Every ❌ line names the exercise and the fix.

In [15]:
review("w06")

w06: 3/3 passed  ·  300/300 marks


True